<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/Lab_3_Modified_e_Greedy_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modified ε-Greedy Strategy for Multi-Armed Bandit

In [1]:
import random

# Part A – Machine Initialization

# Number of machines
M = 3
# Machine names
machines = ['A', 'B', 'C']
# Number of plays
N_play = 10
# Exploration probability
epsilon = 0.2

# Initialize N, R, and Q tables as dictionaries for easy access by machine name
N = {machine: 0 for machine in machines}  # N(a) - number of times machine (a) has been selected
R = {machine: 0 for machine in machines}  # Total reward obtained from machine (a)
Q = {machine: 0.0 for machine in machines} # Q(a) - current estimated value of machine (a)

print("Initial State:")
print(f"N: {N}")
print(f"R: {R}")
print(f"Q: {Q}")

Initial State:
N: {'A': 0, 'B': 0, 'C': 0}
R: {'A': 0, 'B': 0, 'C': 0}
Q: {'A': 0.0, 'B': 0.0, 'C': 0.0}


In [2]:
# Pre-defined reward patterns for simulating user input based on the 'Suggested Experiment'
reward_patterns = {
    'A': [3, 4, 2, 5],
    'B': [6, 7, 5, 8],
    'C': [4, 3, 5, 4]
}

# Keep track of the index for the next reward in each pattern
reward_indices = {machine: 0 for machine in machines}

# Total reward collected by the player
total_player_reward = 0

print("\nStarting Multi-Armed Bandit Simulation...")
print("--------------------------------------")

# Main loop for N_play plays
for play_num in range(1, N_play + 1):
    print(f"\n--- Play {play_num} ---")

    # 1. Generate a random number r between 0 and 1.
    r = random.uniform(0, 1)

    selected_machine = None

    # Part C – Modified ε-Greedy Decision
    if r < epsilon: # 2. If r < ε, perform exploration:
        # Find the machine(s) with the highest current Q-value
        max_q_value = max(Q.values())
        highest_q_machines = [m for m, q in Q.items() if q == max_q_value]

        # Get machines for exploration (all except the one with highest Q-value)
        # If there are multiple machines with the same highest Q, exclude all of them for true exploration
        # However, the problem states 'except highest estimated reward (Q) machine' singular.
        # To simplify and follow the literal interpretation, if multiple have max Q, we'll pick one arbitrarily to exclude

        # Let's consider the simplified case where if multiple machines have the same max Q, we still try to explore from others
        # A safer interpretation for "except highest estimated reward (Q) machine" for true exploration is:
        # if there's a unique highest Q-machine, explore from the rest.
        # If all machines have the same Q (e.g., all 0.0 initially), then exploration is random among all.

        # Handle initial state or ties where all machines might have the same Q value
        if len(highest_q_machines) == M or all(q == 0.0 for q in Q.values()):
             # If all machines have the same Q-value, or all are 0, explore from all machines
            selected_machine = random.choice(machines)
            print(f"Exploration: All Q-values are equal or highest Q-machines cover all. Randomly selected {selected_machine}")
        else:
            # Randomly select one of the three machines except highest estimated reward (Q) machine
            # We need to exclude the machine(s) that have the absolute maximum Q-value
            explorable_machines = [m for m in machines if Q[m] < max_q_value]
            if explorable_machines: # Ensure there are machines to explore from
                selected_machine = random.choice(explorable_machines)
                print(f"Exploration: Highest Q-machine(s) are {highest_q_machines}. Randomly selected {selected_machine} from remaining.")
            else:
                # Fallback: if somehow all machines have max_q_value (shouldn't happen with the `if len(highest_q_machines) == M` check)
                # or if explorable_machines is empty, just pick randomly from all.
                selected_machine = random.choice(machines)
                print(f"Exploration fallback: Could not find distinct explorable machines. Randomly selected {selected_machine}.")

    else: # 3. Otherwise, perform exploitation:
        # Select the machine having the highest current Q-value.
        # If there's a tie, random.choice will pick one among them.
        max_q_value = max(Q.values())
        best_machines = [m for m, q in Q.items() if q == max_q_value]
        selected_machine = random.choice(best_machines) # Choose randomly among the best if ties exist
        print(f"Exploitation: Selected {selected_machine} (highest Q-value: {Q[selected_machine]:.2f})")

    print(f"Play with machine {selected_machine}")

    # Part D – Reward Collection (Simulate user input from pre-defined patterns)
    if reward_indices[selected_machine] < len(reward_patterns[selected_machine]):
        reward = reward_patterns[selected_machine][reward_indices[selected_machine]]
        reward_indices[selected_machine] += 1
    else:
        # If pattern exhausted, just use the last reward or a default (e.g., 0 for simplicity, or repeat last)
        # For this problem, the patterns are short and might be exhausted within 10 plays.
        # Let's assume the pattern repeats or use the last value for simplicity if it runs out.
        # The problem statement's suggested experiment has 4 rewards for each, and N_play = 10, so it will repeat.
        last_idx = len(reward_patterns[selected_machine]) - 1
        reward = reward_patterns[selected_machine][last_idx]
        print(f"Warning: Reward pattern for {selected_machine} exhausted. Using last reward: {reward}")

    print(f"Obtained reward: {reward}")
    total_player_reward += reward

    # Part E – Q-Value Update
    # 5. Update the number of times the selected machine has been played.
    N[selected_machine] += 1

    # Update total reward for the machine (not explicitly asked in formula but good for tracking)
    R[selected_machine] += reward

    # 6. Update its Q-value using the incremental sample-average formula
    # Q_new(a) = Q(a) + (1/N(a)) * (R - Q(a))
    # R here refers to the *current* reward, not total R
    if N[selected_machine] > 0: # Avoid division by zero
        Q[selected_machine] = Q[selected_machine] + (1 / N[selected_machine]) * (reward - Q[selected_machine])

    print(f"Updated N: {N}")
    print(f"Updated Q: {Q}")
    print("--------------------------------------")


Starting Multi-Armed Bandit Simulation...
--------------------------------------

--- Play 1 ---
Exploitation: Selected B (highest Q-value: 0.00)
Play with machine B
Obtained reward: 6
Updated N: {'A': 0, 'B': 1, 'C': 0}
Updated Q: {'A': 0.0, 'B': 6.0, 'C': 0.0}
--------------------------------------

--- Play 2 ---
Exploitation: Selected B (highest Q-value: 6.00)
Play with machine B
Obtained reward: 7
Updated N: {'A': 0, 'B': 2, 'C': 0}
Updated Q: {'A': 0.0, 'B': 6.5, 'C': 0.0}
--------------------------------------

--- Play 3 ---
Exploitation: Selected B (highest Q-value: 6.50)
Play with machine B
Obtained reward: 5
Updated N: {'A': 0, 'B': 3, 'C': 0}
Updated Q: {'A': 0.0, 'B': 6.0, 'C': 0.0}
--------------------------------------

--- Play 4 ---
Exploration: Highest Q-machine(s) are ['B']. Randomly selected A from remaining.
Play with machine A
Obtained reward: 3
Updated N: {'A': 1, 'B': 3, 'C': 0}
Updated Q: {'A': 3.0, 'B': 6.0, 'C': 0.0}
--------------------------------------

-

## Part F – Display Results

In [3]:
print("\n--- Final Results After All Plays ---")

print("Participation Count (N):")
for machine, count in N.items():
    print(f"  Machine {machine}: {count} plays")

print("\nTotal Rewards Obtained (R):")
for machine, total_r in R.items():
    print(f"  Machine {machine}: {total_r} total reward")

print("\nEstimated Q-Values:")
for machine, q_val in Q.items():
    print(f"  Machine {machine}: {q_val:.2f}")

print(f"\nTotal reward the player received: {total_player_reward}")


--- Final Results After All Plays ---
Participation Count (N):
  Machine A: 1 plays
  Machine B: 9 plays
  Machine C: 0 plays

Total Rewards Obtained (R):
  Machine A: 3 total reward
  Machine B: 66 total reward
  Machine C: 0 total reward

Estimated Q-Values:
  Machine A: 3.00
  Machine B: 7.33
  Machine C: 0.00

Total reward the player received: 69


## Submission Requirements: Calculations for Two Plays

The detailed steps for each play, including the selection logic, reward obtained, and updated N and Q values, are printed within the main simulation loop above. You can refer to the output of the previous cell for the calculations for the first two plays (and subsequent ones).